# NFHS-5: Does a screening model trained on urban women work for rural women?

**Question:** If we build a model on **urban** women to predict who has had a cervical cancer screening,
does it still work for **rural** women? Or does it get worse, and does it over- or under-estimate their screening?

**Data rule (DHS terms):** the `.DTA` and `.pkl` files stay on this laptop. Never upload them to Claude, GitHub or anywhere else.
Share only the **printed tables** and the **PNG plot**.

**How to run:** run the cells from top to bottom. *Settings* must run first every time you open the notebook.
Cell 2 is the slow one (a few minutes). After that you can skip Cells 1–2, because Cell 3 loads the small saved file.

In [ ]:
# SETTINGS: run this first, every time you open the notebook
import os, time
import numpy as np
import pandas as pd

FOLDER  = r"D:\DHS dataset"                                   # your data folder (stays on your laptop)
DTA     = os.path.join(FOLDER, "IAIR7EFL.DTA")                # NFHS-5 women's file
SUBSET  = os.path.join(FOLDER, "nfhs5_transfer_30_49.pkl")    # smaller file created in Cell 2
RESULTS = os.path.join(FOLDER, "results")                     # summary tables and plots only
os.makedirs(RESULTS, exist_ok=True)

OUTCOME = "sb79"     # "sb79" = cervical screening (main). Later you can try "sb80" = breast screening.
SCREEN  = {"sb79": "cervical", "sb80": "breast"}[OUTCOME]
pd.set_option("display.width", 220)
print(f"Settings ready. Outcome: {OUTCOME} ({SCREEN} cancer screening)")

## Cell 1: Check the variable names
This reads only the **list of variable names and labels** (not the data), so it is quick.
It confirms that each variable we plan to use exists and means what we think.

In [ ]:
# CELL 1: check variable names (reads the file header only)
with pd.read_stata(DTA, iterator=True) as reader:
    labels = reader.variable_labels()

planned = ["v005", "v021", "v022", "v024", "v025", "v012", "v106", "v130", "v190", "v201",
           "v481", "v501", "v157", "v158", "v159", "v169a", "v171a", "v393", "v394",
           "s116", "sb79", "sb80"]
for v in planned:
    print(f"{v:6s} -> {labels.get(v, '*** NOT FOUND ***')}")

print("\nOther variables whose label mentions caste, tribe, health worker or insurance:")
keywords = ["caste", "tribe", "health worker", "fieldworker", "asha", "anm", "insurance"]
for name, lab in labels.items():
    if any(k in lab.lower() for k in keywords) and name not in planned:
        print(f"{name:8s} -> {lab}")

## Cell 2: Load only what we need (women aged 30–49) and save a small file
Same method as your Step 1: read the big file in pieces of 100,000 rows. Each piece is filtered
to ages 30–49 straight away, to save memory. **This takes a few minutes.**

In [ ]:
# CELL 2: load the needed variables for women aged 30-49 and save a smaller file
t0 = time.time()
with pd.read_stata(DTA, iterator=True) as reader:
    available = set(reader.variable_labels())

cols = ["v005", "v021", "v022", "v024", "v025", "v012", "v106", "v130", "v190", "v201",
        "v481", "v501", "v157", "v158", "v159", "v169a", "v171a", "v393", "v394",
        "s116", "sb79", "sb80"]
skipped = [c for c in cols if c not in available]
cols = [c for c in cols if c in available]
print("Not in the file (skipped):", skipped if skipped else "none")

parts = []
for chunk in pd.read_stata(DTA, columns=cols, convert_categoricals=False, chunksize=100_000):
    parts.append(chunk[(chunk["v012"] >= 30) & (chunk["v012"] <= 49)])
df = pd.concat(parts, ignore_index=True)
df.to_pickle(SUBSET)

print(f"Women aged 30-49: {len(df):,}   (took {time.time() - t0:.0f} seconds)")
print("\nShare of missing values per variable:")
print(df.isna().mean().round(3).to_string())

## Cell 3: The main experiment
1. **Outcome:** ever screened for cervical cancer (`sb79 = 1`), set by `OUTCOME` in Settings.
2. **Split by cluster** (village / urban block, `v021`): women from the same cluster are never split between
   training and test. They are similar to each other, so splitting them would leak information, like your run-walk lesson.
3. **Same training size** for the urban and rural models, so any difference is not just "more data".
4. **Two models:** logistic regression (simple, explainable) and gradient boosting (flexible).
5. **Weights (`v005`)** are used in training and in every metric, so the results describe the population, not just the sample.
6. **Compare on the same rural test women:** a model trained on rural women (home) vs a model trained on urban women (transfer).

In [ ]:
# CELL 3: train on urban vs rural women, test on rural women
from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

df = pd.read_pickle(SUBSET)
d = df.dropna(subset=[OUTCOME]).copy()
d["y"] = (d[OUTCOME] == 1).astype(int)          # 1 = ever screened
d["w"] = d["v005"] / 1_000_000                  # survey weight

num_cols = [c for c in ["v012", "v201"] if c in d.columns and d[c].notna().any()]   # age, children ever born
cat_cols = [c for c in ["v024", "v106", "v130", "v190", "s116", "v481", "v501",
                        "v157", "v158", "v159", "v169a", "v171a", "v393", "v394"]
            if c in d.columns and d[c].notna().any()]
features = num_cols + cat_cols
print("Predictors used:", features)

urban = d[d["v025"] == 1]
rural = d[d["v025"] == 2]

def split_by_cluster(data, test_size=0.3, seed=42):
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    train_idx, test_idx = next(gss.split(data, groups=data["v021"]))
    return data.iloc[train_idx], data.iloc[test_idx]

urban_train, urban_test = split_by_cluster(urban)
rural_train_all, rural_test = split_by_cluster(rural)

n_train = min(len(urban_train), len(rural_train_all))       # equal training size
urban_train = urban_train.sample(n=n_train, random_state=0)
rural_train = rural_train_all.sample(n=n_train, random_state=0)

print(f"\nTraining women per model: {n_train:,}  "
      f"(screened: urban {urban_train['y'].sum():,}, rural {rural_train['y'].sum():,})")
print(f"Test women: urban {len(urban_test):,} (screened {urban_test['y'].sum():,}), "
      f"rural {len(rural_test):,} (screened {rural_test['y'].sum():,})")

def make_model(name):
    if name == "Logistic regression":
        prep = ColumnTransformer([
            ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                              ("scale", StandardScaler())]), num_cols),
            ("cat", Pipeline([("impute", SimpleImputer(strategy="constant", fill_value=-1)),
                              ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
        ])
        return Pipeline([("prep", prep), ("clf", LogisticRegression(max_iter=2000))])
    is_cat = [c in cat_cols for c in features]
    return HistGradientBoostingClassifier(categorical_features=is_cat, random_state=0)

def train(name, data):
    X, y = data[features], data["y"]
    w = data["w"] / data["w"].mean()
    model = make_model(name)
    if name == "Logistic regression":
        model.fit(X, y, clf__sample_weight=w)
    else:
        model.fit(X, y, sample_weight=w)
    return model

def evaluate(model, test):
    p = model.predict_proba(test[features])[:, 1]
    y, w = test["y"], test["w"]
    return {"ROC-AUC": roc_auc_score(y, p, sample_weight=w),
            "PR-AUC": average_precision_score(y, p, sample_weight=w),
            "Brier": brier_score_loss(y, p, sample_weight=w),
            "Observed %": 100 * np.average(y, weights=w),
            "Predicted %": 100 * np.average(p, weights=w)}, p

MODELS = ["Logistic regression", "Gradient boosting"]
rows, preds = [], {}
for name in MODELS:
    m_urban = train(name, urban_train)
    m_rural = train(name, rural_train)
    for setting, model, test in [("Urban -> Urban (home)", m_urban, urban_test),
                                 ("Rural -> Rural (home)", m_rural, rural_test),
                                 ("Urban -> Rural (transfer)", m_urban, rural_test)]:
        scores, p = evaluate(model, test)
        rows.append({"Model": name, "Train -> Test": setting, **scores})
        preds[(name, setting)] = p

results = pd.DataFrame(rows)
results["Predicted / observed"] = results["Predicted %"] / results["Observed %"]
print("\n" + results.round(3).to_string(index=False))
results.round(4).to_csv(os.path.join(RESULTS, f"transfer_results_{OUTCOME}.csv"), index=False)

print("\nIn plain words:")
r = results.set_index(["Model", "Train -> Test"])
for name in MODELS:
    home, tr = r.loc[(name, "Rural -> Rural (home)")], r.loc[(name, "Urban -> Rural (transfer)")]
    print(f"- {name}: on rural women, ROC-AUC {tr['ROC-AUC']:.3f} if trained on urban women vs "
          f"{home['ROC-AUC']:.3f} if trained on rural women. The urban-trained model predicts "
          f"{tr['Predicted %']:.2f}% screened; the real figure is {tr['Observed %']:.2f}%.")

## Cell 4: How sure are we? (cluster bootstrap)
One test set gives one number. To see how much that number could move by chance, we redraw the rural test
**clusters** with replacement 500 times and recompute everything. The 95% interval is the middle 95% of those
500 results. If the interval for the **AUC drop** does not include 0, the drop is unlikely to be due to chance.
(Takes about 1–3 minutes.)

In [ ]:
# CELL 4: 95% intervals by resampling rural test clusters (500 times)
t0 = time.time()
rng = np.random.default_rng(0)
groups = rural_test["v021"].to_numpy()
order = np.argsort(groups, kind="stable")
_, starts = np.unique(groups[order], return_index=True)
cluster_rows = np.split(order, starts[1:])            # row positions of each cluster
y_t, w_t = rural_test["y"].to_numpy(), rural_test["w"].to_numpy()

def stats(idx, name):
    yy, ww = y_t[idx], w_t[idx]
    p_home = preds[(name, "Rural -> Rural (home)")][idx]
    p_tr = preds[(name, "Urban -> Rural (transfer)")][idx]
    obs = np.average(yy, weights=ww)
    auc_home = roc_auc_score(yy, p_home, sample_weight=ww)
    auc_tr = roc_auc_score(yy, p_tr, sample_weight=ww)
    return [auc_home, auc_tr, auc_home - auc_tr,
            np.average(p_home, weights=ww) / obs, np.average(p_tr, weights=ww) / obs]

B = 500
boot = {name: [] for name in MODELS}
for b in range(B):
    idx = np.concatenate([cluster_rows[k] for k in rng.integers(0, len(cluster_rows), len(cluster_rows))])
    for name in MODELS:
        boot[name].append(stats(idx, name))

labels_ = ["ROC-AUC, trained on rural (home)", "ROC-AUC, trained on urban (transfer)",
           "AUC drop (home minus transfer)", "Predicted/observed, trained on rural",
           "Predicted/observed, trained on urban"]
everyone = np.arange(len(y_t))
table = {}
for name in MODELS:
    point = stats(everyone, name)
    lo, hi = np.percentile(np.array(boot[name]), [2.5, 97.5], axis=0)
    table[name] = [f"{p:.3f}  [{l:.3f}, {h:.3f}]" for p, l, h in zip(point, lo, hi)]
ci_table = pd.DataFrame(table, index=labels_)
print("Rural test women: estimate [95% interval]\n")
print(ci_table.to_string())
ci_table.to_csv(os.path.join(RESULTS, f"transfer_bootstrap_95ci_{OUTCOME}.csv"))
print(f"\n({B} resamples of {len(cluster_rows):,} rural test clusters, {time.time() - t0:.0f} seconds)")

## Cell 5: Calibration picture
Rural test women are sorted by predicted risk and split into 10 equal groups. For each group we compare the
**average predicted** screening rate (x-axis) with the **real** rate (y-axis).
Points on the dashed line mean the predictions match reality. Points **below** the line mean the model **over-estimates** screening.

In [ ]:
# CELL 5: calibration plot for rural test women (saved as PNG)
import matplotlib.pyplot as plt

def tenths(p, y, w, n_bins=10):
    t = pd.DataFrame({"p": p, "y": y, "w": w})
    t["group"] = pd.qcut(t["p"].rank(method="first"), n_bins, labels=False) + 1
    t["pw"], t["yw"] = t["p"] * t["w"], t["y"] * t["w"]
    g = t.groupby("group")[["pw", "yw", "w"]].sum()
    return pd.DataFrame({"Predicted %": 100 * g["pw"] / g["w"],
                         "Observed %": 100 * g["yw"] / g["w"],
                         "Women": t.groupby("group").size()})

BLUE, ORANGE = "#2a78d6", "#eb6834"                 # colour-blind-safe pair
INK, SOFT, MUTED, GRID, AXIS = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"

tabs = {}
for name in MODELS:
    tabs[(name, "rural")] = tenths(preds[(name, "Rural -> Rural (home)")], y_t, w_t)
    tabs[(name, "urban")] = tenths(preds[(name, "Urban -> Rural (transfer)")], y_t, w_t)
top = 1.08 * max(t[["Predicted %", "Observed %"]].to_numpy().max() for t in tabs.values())

fig, axes = plt.subplots(1, 2, figsize=(11, 4.8), sharex=True, sharey=True)
for ax, name in zip(axes, MODELS):
    ax.plot([0, top], [0, top], ls="--", lw=1, color=MUTED, label="Perfect match")
    for key, colour, marker, text in [("rural", BLUE, "o", "Trained on rural women"),
                                      ("urban", ORANGE, "s", "Trained on urban women")]:
        t = tabs[(name, key)]
        ax.plot(t["Predicted %"], t["Observed %"], marker=marker, color=colour, lw=2, ms=7,
                markeredgecolor="white", markeredgewidth=1.2, label=text)
    ax.set_title(name, fontsize=11, color=INK, loc="left")
    ax.set_xlim(0, top); ax.set_ylim(0, top)
    ax.set_xlabel("Average predicted screening (%)", color=SOFT)
    ax.grid(color=GRID, lw=0.6); ax.set_axisbelow(True)
    for side in ["top", "right"]:
        ax.spines[side].set_visible(False)
    for side in ["left", "bottom"]:
        ax.spines[side].set_color(AXIS)
    ax.tick_params(colors=SOFT, labelsize=9)
axes[0].set_ylabel("Observed screening (%)", color=SOFT)
axes[0].legend(frameon=False, fontsize=9, loc="upper left", labelcolor=INK)
fig.suptitle(f"Rural test women: predicted vs observed {SCREEN} screening, by tenth of predicted risk",
             fontsize=12, color=INK, x=0.01, ha="left")
fig.tight_layout()
fig.savefig(os.path.join(RESULTS, f"calibration_urban_to_rural_{OUTCOME}.png"), dpi=150, bbox_inches="tight")
plt.show()

for name in MODELS:
    both = pd.concat({"Trained on rural": tabs[(name, "rural")],
                      "Trained on urban": tabs[(name, "urban")]}, axis=1).round(2)
    print(f"\n{name}: rural test women by tenth of predicted risk (1 = lowest)")
    print(both.to_string())

## What to send Claude
Send **only** these. They are summaries, which the DHS terms allow:
1. The printed output of **Cell 1** (variable names and labels)
2. The printed output of **Cell 3** (results table and "In plain words")
3. The printed output of **Cell 4** (95% intervals)
4. The plot **`calibration_urban_to_rural_sb79.png`** from the `results` folder

Never upload the `.DTA` or `.pkl` files.